# American Express Credit Card Default Prediction — Dataset Exploration

This notebook explores the **American Express Default Prediction** dataset  
(Kaggle competition: `amex-default-prediction`).

### Dataset overview
| Attribute | Detail |
|-----------|--------|
| **Task** | Binary classification — predict customer default (label = 1) |
| **Rows** | ~5.5 M statement rows (train) spanning ~460 K unique customers |
| **Features** | 190 anonymised features across 5 groups: **S** spend, **P** payment, **D** delinquency, **B** balance, **R** risk |
| **Time steps** | Up to 13 monthly statements per customer |
| **Evaluation metric** | Normalized Gini + D-score: $M = 0.5 (G + D)$ |
| **Kaggle page** | https://www.kaggle.com/competitions/amex-default-prediction |

> **Data access** — Download via Kaggle API (cell 2) or place the files manually  
> under `data/amex/` (see instructions below).


## 1. Import Required Libraries

In [ ]:
from __future__ import annotations

# ── Standard library ────────────────────────────────────────────────────────
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Data manipulation ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Machine learning ─────────────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

# Optional: LightGBM (install if missing)
try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except ImportError:
    LGB_AVAILABLE = False
    print("LightGBM not installed — run `pip install lightgbm` to enable model training.")

# ── Plot style ───────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 5)})

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("..").resolve()
AMEX_DIR     = PROJECT_ROOT / "data" / "amex"
AMEX_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"AmEx data dir: {AMEX_DIR}")


## 2. Download / Load the Dataset

### Option A — Kaggle API (recommended)
```bash
pip install kaggle
# Place kaggle.json in ~/.kaggle/  (from kaggle.com → Account → API → Create New Token)
kaggle competitions download -c amex-default-prediction -p data/amex/
cd data/amex && unzip amex-default-prediction.zip
```

### Option B — Manual download
1. Go to https://www.kaggle.com/competitions/amex-default-prediction/data  
2. Download `train_data.csv`, `train_labels.csv`, `test_data.csv`  
3. Place them in `data/amex/`

The cells below detect whichever format is present and load accordingly.  
A **5 000-customer sample** is used for interactive EDA to keep memory manageable;  
set `SAMPLE_CUSTOMERS = None` to load everything.

In [ ]:
def _download_via_kaggle(dest: Path) -> None:
    """Attempt to pull competition data with the Kaggle CLI."""
    import subprocess
    print("Downloading AmEx dataset via Kaggle API …")
    subprocess.run(
        ["kaggle", "competitions", "download",
         "-c", "amex-default-prediction", "-p", str(dest)],
        check=True,
    )
    import zipfile
    for zf in dest.glob("*.zip"):
        with zipfile.ZipFile(zf) as z:
            z.extractall(dest)
        zf.unlink()
    print("Download complete.")


# ── Locate raw files ──────────────────────────────────────────────────────────
TRAIN_CSV   = AMEX_DIR / "train_data.csv"
LABEL_CSV   = AMEX_DIR / "train_labels.csv"
TRAIN_PARQ  = AMEX_DIR / "train_data.parquet"  # faster if already converted

if not TRAIN_CSV.exists() and not TRAIN_PARQ.exists():
    try:
        _download_via_kaggle(AMEX_DIR)
    except Exception as e:
        print(f"Auto-download failed ({e}).")
        print("Please place train_data.csv + train_labels.csv in:", AMEX_DIR)

# ── Load ──────────────────────────────────────────────────────────────────────
SAMPLE_CUSTOMERS: int | None = 5_000   # set None for the full ~460 K customers

if TRAIN_PARQ.exists():
    raw = pd.read_parquet(TRAIN_PARQ)
    print("Loaded from parquet cache.")
elif TRAIN_CSV.exists():
    # First 5 K customers → use nrows heuristic (≈ 13 rows each)
    nrows = SAMPLE_CUSTOMERS * 14 if SAMPLE_CUSTOMERS else None
    raw = pd.read_csv(TRAIN_CSV, nrows=nrows, low_memory=False)
    if SAMPLE_CUSTOMERS:
        keep = raw["customer_ID"].unique()[:SAMPLE_CUSTOMERS]
        raw  = raw[raw["customer_ID"].isin(keep)].reset_index(drop=True)
    # Cache for speed
    raw.to_parquet(TRAIN_PARQ, index=False)
    print("Loaded from CSV and cached to parquet.")
else:
    print("⚠️  No data files found. Using a tiny synthetic stand-in for demo purposes.")
    # Build a minimal synthetic AmEx-shaped DataFrame for demonstration
    rng = np.random.default_rng(0)
    n   = 1_300   # 100 customers × ~13 statements
    cids = np.repeat([f"C{i:04d}" for i in range(100)], 13)
    dates = pd.date_range("2017-03-01", periods=13, freq="MS")
    dates = np.tile(dates, 100)
    feat_cols = (
        [f"P_{i}" for i in range(1, 5)]    # payment
        + [f"D_{i}" for i in range(1, 10)] # delinquency
        + [f"S_{i}" for i in range(1, 5)]  # spend
        + [f"B_{i}" for i in range(1, 5)]  # balance
        + [f"R_{i}" for i in range(1, 4)]  # risk
    )
    raw = pd.DataFrame({"customer_ID": cids, "S_2": dates})
    for col in feat_cols:
        raw[col] = rng.normal(size=n)
    # Inject some nulls at 15 % rate
    mask = rng.random(raw[feat_cols].shape) < 0.15
    raw[feat_cols] = raw[feat_cols].where(~mask)
    print("Synthetic demo data created (100 customers × 13 statements).")

if LABEL_CSV.exists():
    labels = pd.read_csv(LABEL_CSV)
else:
    # Synthetic labels aligned to demo customers
    unique_cids = raw["customer_ID"].unique()
    rng2 = np.random.default_rng(1)
    y    = (rng2.random(len(unique_cids)) < 0.26).astype(int)
    labels = pd.DataFrame({"customer_ID": unique_cids, "target": y})
    print("Synthetic labels created.")

print(f"\nRaw shape : {raw.shape}")
print(f"Unique customers : {raw['customer_ID'].nunique():,}")
print(f"Label shape : {labels.shape}  |  default rate: {labels['target'].mean():.2%}")


## 3. Load and Inspect the Dataset

In [ ]:
# ── Feature groups ─────────────────────────────────────────────────────────
NON_FEAT_COLS = {"customer_ID", "S_2"}
ALL_FEAT_COLS = [c for c in raw.columns if c not in NON_FEAT_COLS]

CAT_FEATURES  = ["B_30", "B_38", "D_114", "D_116", "D_117", "D_120",
                 "D_126", "D_63", "D_64", "D_66", "D_68"]
CAT_FEATURES  = [c for c in CAT_FEATURES if c in raw.columns]
NUM_FEATURES  = [c for c in ALL_FEAT_COLS if c not in CAT_FEATURES]

FEAT_GROUPS = {
    "S (spend)":       [c for c in ALL_FEAT_COLS if c.startswith("S_") and c != "S_2"],
    "P (payment)":     [c for c in ALL_FEAT_COLS if c.startswith("P_")],
    "D (delinquency)": [c for c in ALL_FEAT_COLS if c.startswith("D_")],
    "B (balance)":     [c for c in ALL_FEAT_COLS if c.startswith("B_")],
    "R (risk)":        [c for c in ALL_FEAT_COLS if c.startswith("R_")],
}

print("=== Dataset Shape ===")
print(f"  Rows × Cols : {raw.shape}")
print(f"  Unique customers  : {raw['customer_ID'].nunique():,}")
print(f"  Statements per customer (avg): {len(raw)/raw['customer_ID'].nunique():.1f}")
print(f"\n=== Feature Groups ===")
for grp, cols in FEAT_GROUPS.items():
    print(f"  {grp:<20} : {len(cols):>3} features")
print(f"  {'Categorical':20} : {len(CAT_FEATURES):>3} features")
print(f"  {'Numerical':20} : {len(NUM_FEATURES):>3} features")

print("\n=== First 3 rows ===")
display(raw.head(3))

print("\n=== DTypes ===")
display(raw.dtypes.value_counts().rename("count").to_frame())


## 4. Data Cleaning and Preprocessing

In [ ]:
# ── Missing-value analysis ─────────────────────────────────────────────────
miss = (
    raw[ALL_FEAT_COLS]
    .isnull()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .to_frame()
)
miss["group"] = miss.index.map(lambda c: c[0])

HIGH_MISS_THRESH = 0.30
high_miss = miss[miss["missing_rate"] >= HIGH_MISS_THRESH]
print(f"Features with ≥{HIGH_MISS_THRESH:.0%} missing: {len(high_miss)}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: overall distribution of missing rates
axes[0].hist(miss["missing_rate"], bins=30, color="#4878CF", edgecolor="white")
axes[0].set_title("Distribution of per-feature missing rates")
axes[0].set_xlabel("Fraction missing")
axes[0].set_ylabel("# features")
axes[0].axvline(HIGH_MISS_THRESH, color="red", linestyle="--", label=f"≥{HIGH_MISS_THRESH:.0%} threshold")
axes[0].legend()

# Right: mean missing rate per feature group
miss.groupby("group")["missing_rate"].mean().sort_values().plot.barh(ax=axes[1], color="#6ACC65")
axes[1].set_title("Mean missing rate by feature group")
axes[1].set_xlabel("Avg fraction missing")
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.show()


In [ ]:
# ── Fix dtypes ─────────────────────────────────────────────────────────────
df = raw.copy()
df["S_2"] = pd.to_datetime(df["S_2"])

# Ordinal-encode categoricals (AmEx cats are already numeric; coerce anyway)
for col in CAT_FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int8")

# Numeric columns → float32 to halve memory
for col in NUM_FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

# Drop exact duplicate statement rows (rare but possible in raw data)
before = len(df)
df = df.drop_duplicates(subset=["customer_ID", "S_2"]).reset_index(drop=True)
print(f"Duplicate statement rows removed: {before - len(df)}")

# Merge labels
df = df.merge(labels, on="customer_ID", how="left")

print(f"\nCleaned shape: {df.shape}")
print(f"Memory usage : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
display(df.describe(include="all").T.head(10))


## 5. Exploratory Data Analysis (EDA)

### 5a. Class balance

In [ ]:
cust_labels = labels.copy()
default_rate = cust_labels["target"].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Pie chart
axes[0].pie(
    [1 - default_rate, default_rate],
    labels=["Non-default", "Default"],
    autopct="%1.1f%%",
    colors=["#4878CF", "#D65F5F"],
    startangle=90,
)
axes[0].set_title("Class distribution (customer level)")

# Statements per customer
stmt_counts = df.groupby("customer_ID").size()
axes[1].hist(stmt_counts, bins=range(1, 16), color="#6ACC65", edgecolor="white", align="left")
axes[1].set_title("Statements per customer")
axes[1].set_xlabel("# monthly statements")
axes[1].set_ylabel("# customers")
axes[1].set_xticks(range(1, 15))

plt.tight_layout()
plt.show()

print(f"Default rate    : {default_rate:.2%}")
print(f"Avg statements  : {stmt_counts.mean():.2f}  |  median: {stmt_counts.median():.0f}")


### 5b. Key feature distributions (P_2, D_39, B_1, R_1 — one from each group)

In [ ]:
PROBE_COLS = [c for c in ["P_2", "D_39", "B_1", "R_1"] if c in df.columns]
if not PROBE_COLS:
    # fallback to first available numerical features
    PROBE_COLS = NUM_FEATURES[:4]

df_probe = df[PROBE_COLS + ["target"]].dropna(subset=PROBE_COLS[:1])

fig, axes = plt.subplots(2, len(PROBE_COLS), figsize=(5 * len(PROBE_COLS), 8))

for i, col in enumerate(PROBE_COLS):
    # Top row: histogram by target
    for tgt, clr in [(0, "#4878CF"), (1, "#D65F5F")]:
        subset = df_probe[df_probe["target"] == tgt][col].dropna()
        axes[0, i].hist(subset, bins=50, alpha=0.6, color=clr,
                        label=f"target={tgt}", density=True)
    axes[0, i].set_title(f"{col} — histogram by default")
    axes[0, i].legend(fontsize=8)

    # Bottom row: box plot by target
    df_probe.boxplot(column=col, by="target", ax=axes[1, i],
                     showfliers=False, patch_artist=True,
                     boxprops=dict(facecolor="#4878CF", alpha=0.5))
    axes[1, i].set_title(f"{col} — boxplot by default")
    axes[1, i].set_xlabel("target")

plt.suptitle("Key feature distributions split by default label", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 6. Transaction Pattern Analysis

AmEx statements span up to 13 months (2017-03 → 2018-04 in the competition data).  
Here we look at volume trends over time and statement-count patterns by default class.

In [ ]:
df["year_month"] = df["S_2"].dt.to_period("M")

# ── Monthly statement volume ────────────────────────────────────────────────
monthly_vol = df.groupby("year_month").size().rename("statements")

# ── Avg key metric over time (P_2 = payment feature if present) ────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

monthly_vol.plot(ax=axes[0], marker="o", color="#4878CF")
axes[0].set_title("Monthly statement volume")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("# statements")
axes[0].tick_params(axis="x", rotation=45)

# Statements per customer split by default class
df_with_target = df.dropna(subset=["target"])
stmt_by_class  = (
    df_with_target
    .groupby(["customer_ID", "target"])
    .size()
    .reset_index(name="stmt_count")
)
for tgt, grp in stmt_by_class.groupby("target"):
    axes[1].hist(grp["stmt_count"], bins=range(1, 16), alpha=0.65,
                 label=f"target={int(tgt)}", align="left", density=True,
                 color="#D65F5F" if tgt == 1 else "#4878CF")
axes[1].set_title("Statements per customer by default class")
axes[1].set_xlabel("# monthly statements")
axes[1].set_ylabel("Density")
axes[1].legend()
axes[1].set_xticks(range(1, 15))

plt.tight_layout()
plt.show()

# ── Trend in a payment feature over time ───────────────────────────────────
if "P_2" in df.columns:
    trend = df.groupby("year_month")["P_2"].mean()
    fig2, ax2 = plt.subplots(figsize=(13, 3))
    trend.plot(ax=ax2, marker="s", color="#6ACC65")
    ax2.set_title("Monthly mean of P_2 (payment feature) — trend over time")
    ax2.set_xlabel("Month")
    ax2.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()


## 7. Feature Engineering

Standard AmEx time-series aggregation strategy:
- **last** — most recent statement value (highest signal for current state)
- **mean / std / min / max** — customer-level statistics across all 13 statements
- **trend** — last − first (direction of change)
- For categoricals: **last** and **mode** (most frequent value)

The result is one row per customer with `~190 × 5 = ~950` features.

In [ ]:
def aggregate_amex_features(
    data: pd.DataFrame,
    num_cols: list[str],
    cat_cols: list[str],
    id_col: str = "customer_ID",
    date_col: str = "S_2",
) -> pd.DataFrame:
    """
    Aggregate time-series statements to one row per customer.

    Numerical cols: last, mean, std, min, max, trend (last - first)
    Categorical cols: last, nunique (cardinality across statements)
    """
    # Sort chronologically within each customer
    data = data.sort_values([id_col, date_col])

    num_cols_present = [c for c in num_cols if c in data.columns]
    cat_cols_present = [c for c in cat_cols if c in data.columns]

    agg_num = {}
    for col in num_cols_present:
        grp = data.groupby(id_col)[col]
        agg_num[f"{col}_last"]  = grp.last()
        agg_num[f"{col}_mean"]  = grp.mean()
        agg_num[f"{col}_std"]   = grp.std()
        agg_num[f"{col}_min"]   = grp.min()
        agg_num[f"{col}_max"]   = grp.max()
        agg_num[f"{col}_trend"] = grp.last() - grp.first()

    agg_cat = {}
    for col in cat_cols_present:
        grp = data.groupby(id_col)[col]
        agg_cat[f"{col}_last"]    = grp.last()
        agg_cat[f"{col}_nunique"] = grp.nunique()

    df_num = pd.DataFrame(agg_num)
    df_cat = pd.DataFrame(agg_cat)
    result = pd.concat([df_num, df_cat], axis=1).reset_index()
    result.columns.name = None
    return result


print("Aggregating features …")
df_agg = aggregate_amex_features(df, NUM_FEATURES, CAT_FEATURES)
df_agg = df_agg.merge(labels, on="customer_ID", how="left")

print(f"Aggregated shape: {df_agg.shape}")
print(f"  Columns: {df_agg.shape[1] - 2} features + customer_ID + target")
display(df_agg.head(3))


In [ ]:
# ── Additional derived features at customer level ──────────────────────────

# 1. Statement count (proxy for customer tenure in the dataset)
stmts_per_cust = df.groupby("customer_ID").size().rename("stmt_count")
df_agg = df_agg.merge(stmts_per_cust, on="customer_ID")

# 2. Spending velocity = mean(P_2_last) / stmt_count  (if P_2 available)
if "P_2_last" in df_agg.columns:
    df_agg["spending_velocity"] = df_agg["P_2_last"] / df_agg["stmt_count"].clip(1)

# 3. Delinquency momentum = last D_39 minus mean D_39  (worsening trend)
if "D_39_last" in df_agg.columns and "D_39_mean" in df_agg.columns:
    df_agg["delinquency_momentum"] = df_agg["D_39_last"] - df_agg["D_39_mean"]

# 4. Balance utilization trend (B_1_trend / B_1_mean)
if "B_1_trend" in df_agg.columns and "B_1_mean" in df_agg.columns:
    df_agg["balance_utilization_trend"] = (
        df_agg["B_1_trend"] / df_agg["B_1_mean"].replace(0, np.nan)
    ).fillna(0)

DERIVED = ["stmt_count", "spending_velocity",
           "delinquency_momentum", "balance_utilization_trend"]
available_derived = [c for c in DERIVED if c in df_agg.columns]
print("Derived features added:", available_derived)

# ── Describe new features ──────────────────────────────────────────────────
display(df_agg[available_derived].describe().T)


## 8. Correlation Analysis

In [ ]:
# Use top 25 features by absolute correlation with target
feat_cols_for_corr = [
    c for c in df_agg.columns
    if c not in {"customer_ID", "target", "year_month"}
    and df_agg[c].dtype in [np.float32, np.float64, np.int64, np.int32, "float32", "float64"]
]

corr_with_target = (
    df_agg[feat_cols_for_corr + ["target"]]
    .dropna(thresh=int(0.5 * len(df_agg)))
    .corr(numeric_only=True)["target"]
    .drop("target")
    .sort_values(key=abs, ascending=False)
)

top25 = corr_with_target.head(25).index.tolist()

corr_mat = (
    df_agg[top25 + ["target"]]
    .dropna(thresh=int(0.5 * len(df_agg)))
    .corr(numeric_only=True)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left: top positive/negative correlations with target
corr_with_target.head(25).plot.barh(ax=axes[0], color="#4878CF")
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Top 25 features by |correlation| with target")
axes[0].set_xlabel("Pearson r")

# Right: heatmap of top-25 feature × feature correlation
sns.heatmap(
    corr_mat, ax=axes[1],
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.2, annot=False,
)
axes[1].set_title("Top-25 feature correlation heatmap")
axes[1].tick_params(axis="x", rotation=90, labelsize=7)
axes[1].tick_params(axis="y", labelsize=7)

plt.tight_layout()
plt.show()


## 9. Visualizing Spending Trends

In [ ]:
S_COLS = [c for c in df.columns if c.startswith("S_") and c != "S_2"][:6]
P_COLS = [c for c in df.columns if c.startswith("P_")][:6]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ── 1. Monthly mean of S_ features (spending) ──────────────────────────────
if S_COLS:
    s_trend = df.groupby("year_month")[S_COLS].mean()
    s_trend.plot(ax=axes[0, 0], marker=".", legend=True)
    axes[0, 0].set_title("Monthly mean of Spend (S_*) features")
    axes[0, 0].tick_params(axis="x", rotation=45)
    axes[0, 0].legend(fontsize=8, ncol=2)

# ── 2. Monthly mean of P_ features (payments) ──────────────────────────────
if P_COLS:
    p_trend = df.groupby("year_month")[P_COLS].mean()
    p_trend.plot(ax=axes[0, 1], marker=".", legend=True)
    axes[0, 1].set_title("Monthly mean of Payment (P_*) features")
    axes[0, 1].tick_params(axis="x", rotation=45)
    axes[0, 1].legend(fontsize=8, ncol=2)

# ── 3. Default vs non-default spending trend (P_2) ─────────────────────────
if "P_2" in df.columns:
    for tgt, clr in [(0, "#4878CF"), (1, "#D65F5F")]:
        sub = df[df["target"] == tgt].groupby("year_month")["P_2"].mean()
        axes[1, 0].plot(sub.index.astype(str), sub.values,
                        label=f"target={tgt}", color=clr, marker="o")
    axes[1, 0].set_title("Avg P_2 over time: default vs non-default")
    axes[1, 0].legend()
    axes[1, 0].tick_params(axis="x", rotation=45)

# ── 4. Feature group mean absolute value comparison ───────────────────────
group_means = {}
for grp_name, cols in FEAT_GROUPS.items():
    avail = [c for c in cols if c in df.columns]
    if avail:
        group_means[grp_name] = df[avail].abs().mean().mean()

if group_means:
    pd.Series(group_means).sort_values().plot.barh(ax=axes[1, 1], color="#6ACC65")
    axes[1, 1].set_title("Mean absolute feature value by group")
    axes[1, 1].set_xlabel("Mean |value|")

plt.tight_layout()
plt.show()


## 10. LightGBM Baseline with AmEx Competition Metric

The competition uses a custom metric:

$$M = 0.5 \left( G + D \right)$$

where  
$G = \text{Normalized Gini}$ (= $2 \times \text{AUC} - 1$)  
$D = \text{Default-rate recall at top } 4\%$ (fraction of true defaults captured in the top-scoring 4% of customers)

In [ ]:
def amex_metric(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Official AmEx default prediction competition metric.

    M = 0.5 * (Normalized_Gini + D)

    Normalized_Gini = 2 * AUC - 1
    D = recall of true defaults at the top-4% score threshold
    """
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=float)

    # Normalized Gini
    auc = roc_auc_score(y_true, y_pred)
    gini = 2 * auc - 1

    # D-score: recall@4%
    top4_threshold = np.percentile(y_pred, 96)        # top 4 % by score
    top4_mask      = y_pred >= top4_threshold
    d_score        = y_true[top4_mask].sum() / y_true.sum()

    return 0.5 * (gini + d_score)


# ── Quick sanity check ─────────────────────────────────────────────────────
_y = np.array([0, 1, 0, 1, 1, 0, 1, 0])
_p = np.array([0.1, 0.9, 0.2, 0.8, 0.7, 0.3, 0.6, 0.4])
print(f"AmEx metric sanity check: {amex_metric(_y, _p):.4f}  (expect > 0.5)")


In [ ]:
if not LGB_AVAILABLE:
    print("Skipping model training — install lightgbm first.")
else:
    # ── Prepare design matrix ────────────────────────────────────────────────
    meta_cols  = {"customer_ID", "target"}
    model_cols = [c for c in df_agg.columns if c not in meta_cols]

    df_model = df_agg[model_cols + ["target"]].dropna(subset=["target"])

    # Fill remaining NaNs with column median
    df_model = df_model.copy()
    for col in model_cols:
        if df_model[col].isnull().any():
            df_model[col] = df_model[col].fillna(df_model[col].median())

    X = df_model[model_cols].values.astype(np.float32)
    y = df_model["target"].values.astype(int)

    print(f"Design matrix  : {X.shape}")
    print(f"Default rate   : {y.mean():.2%}")

    # ── 3-fold stratified CV ─────────────────────────────────────────────────
    skf    = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    oof    = np.zeros(len(y))

    lgb_params = dict(
        n_estimators   = 300,
        learning_rate  = 0.05,
        num_leaves     = 63,
        max_depth      = 6,
        subsample      = 0.8,
        colsample_bytree = 0.7,
        reg_alpha      = 0.1,
        reg_lambda     = 1.0,
        class_weight   = "balanced",
        random_state   = 42,
        verbose        = -1,
    )

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(X[tr_idx], y[tr_idx],
                  eval_set=[(X[va_idx], y[va_idx])],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                              lgb.log_evaluation(-1)])
        oof[va_idx] = model.predict_proba(X[va_idx])[:, 1]
        fold_score  = amex_metric(y[va_idx], oof[va_idx])
        scores.append(fold_score)
        print(f"  Fold {fold+1}/3 — AmEx M = {fold_score:.4f}")

    oof_score = amex_metric(y, oof)
    print(f"\nOOF AmEx metric (M) : {oof_score:.4f}")
    print(f"CV mean ± std       : {np.mean(scores):.4f} ± {np.std(scores):.4f}")


In [ ]:
if LGB_AVAILABLE:
    # ── OOF score distribution ───────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    for tgt, clr in [(0, "#4878CF"), (1, "#D65F5F")]:
        axes[0].hist(oof[y == tgt], bins=50, alpha=0.6, color=clr,
                     label=f"target={tgt}", density=True)
    axes[0].set_title("OOF predicted probability distribution by class")
    axes[0].set_xlabel("Predicted default probability")
    axes[0].legend()

    # ── Feature importance ───────────────────────────────────────────────────
    imp_df = (
        pd.DataFrame({"feature": model_cols,
                      "importance": model.feature_importances_})
        .sort_values("importance", ascending=False)
        .head(20)
    )
    imp_df.plot.barh(x="feature", y="importance", ax=axes[1],
                     color="#6ACC65", legend=False)
    axes[1].invert_yaxis()
    axes[1].set_title("Top-20 LightGBM feature importances (last fold)")
    axes[1].set_xlabel("Gain importance")

    plt.tight_layout()
    plt.show()


## 11. Platform Integration Plan

### How the AmEx dataset maps to this platform

| Platform component | Current (synthetic loans) | AmEx adaptation |
|---|---|---|
| `data/generate_synthetic_data.py` | 1 row per application | Replace/supplement with `data/amex/` loader |
| `feature_pipeline/features.py` | 11 hand-crafted features | Extend with AmEx time-series aggregator (see `feature_pipeline/amex_features.py`) |
| `models/credit_risk/train.py` | LightGBM on loan features | Retrain on `df_agg` with `amex_metric` as eval metric |
| `monitoring/drift_monitor.py` | Monitors 11 features | Extend to track AmEx feature group drift |
| `explainability/shap_explainer.py` | Works unchanged | Apply SHAP on `df_agg` model |
| `mlflow_config/` | Tracks AUC + KS | Add `amex_m_score` as primary MLflow metric |

### Next steps

1. **`feature_pipeline/amex_features.py`** — production-grade time-series aggregator  
2. **`models/credit_risk/train_amex.py`** — training script with `amex_metric` eval  
3. **`ingestion-api`** — add a `/ingest/amex-statement` endpoint to accept monthly statement payloads  
4. **`monitoring`** — add group-level drift detection (S, P, D, B, R feature groups)  
5. **`dashboard`** — add AmEx score panel alongside existing credit panels